# ETS MARL - Q-Learning Baseline

This notebook trains and analyses a **tabular Q-learning** baseline for the EU ETS multi-agent environment.

The Q-learning agents use:
- **State discretization**: 5 features x 3 bins = 243 discrete states
- **Action profiles**: 6 auction + 4 secondary = predefined strategy templates
- **Standard Q-learning**: $\alpha=0.1$, $\gamma=0.95$, $\varepsilon$-greedy exploration

This serves as a comparison point for the PPO/HAPPO agents.

## 1. Setup

In [ ]:
import os, sys

# Navigate to project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

print(f'Working directory: {os.getcwd()}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import pickle
import time

from src.environment.ets_environment import ETSEnvironment
from src.agents.q_learning_agent import QLearningAgent, StateDiscretizer, ActionProfileMapper
from src.train_qlearning import train_qlearning, evaluate_qlearning, merge_configs
from src.analysis.qlearning_analysis import (
    plot_qtable_heatmaps, plot_secondary_heatmaps,
    plot_strategy_frequency, plot_reward_comparison,
    plot_price_comparison, plot_green_comparison,
    plot_compliance_comparison, load_training_log
)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)

print('All imports successful.')

## 2. Load Configuration

In [ ]:
with open('configs/default.yaml') as f:
    base_config = yaml.safe_load(f)

with open('configs/qlearning.yaml') as f:
    ql_config = yaml.safe_load(f)

config = merge_configs(base_config, ql_config)

n_agents = config['companies']['n_agents']
n_years = config['simulation']['n_years']
ql_params = config['qlearning']

print(f'Agents: {n_agents} learning + {config["companies"].get("n_bot_agents", 0)} bots')
print(f'Episode length: {n_years} years')
print(f'Q-learning: {ql_params["n_episodes"]} episodes, '
      f'alpha={ql_params["alpha"]}, gamma={ql_params["gamma"]}')
print(f'Epsilon: {ql_params["epsilon_start"]} -> {ql_params["epsilon_end"]} '
      f'over {ql_params["epsilon_decay_frac"]*100:.0f}% of training')
print(f'Reward shaping: beta={config["reward"]["shaping_beta"]}, '
      f'gamma={config["reward"]["shaping_gamma"]} (disabled for baseline)')

## 3. Explore the State/Action Space

Before training, let's understand how the discretizer and action profiles work.

In [ ]:
# Show state space structure
disc = StateDiscretizer()
print(f'Total discrete states: {disc.N_STATES}')
print(f'Features and bin edges:')
for name, edges in disc._BINS.items():
    print(f'  {name:10s}: low <= {edges[0]:.2f} | mid <= {edges[1]:.2f} | high')

print(f'\nAction profiles:')
mapper = ActionProfileMapper()
print(f'  Auction ({mapper.N_AUCTION_PROFILES}): {mapper.auction_profile_names()}')
print(f'  Secondary ({mapper.N_SECONDARY_PROFILES}): {mapper.secondary_profile_names()}')
print(f'  Total action combinations: {mapper.N_AUCTION_PROFILES * mapper.N_SECONDARY_PROFILES}')

In [ ]:
# Show what each auction profile produces at a reference price
env_temp = ETSEnvironment(config, seed=42)
env_temp.reset(seed=42)

ref_price = 80.0  # reference MA3 price
company = env_temp.companies[0]  # coal-heavy agent

print(f'Auction profiles at MA3 price = {ref_price} EUR/t (Agent A1: coal-heavy):')
print(f'{"Profile":<15s} {"Bid":>7s} {"Qty":>5s} {"Inv%":>5s} {"Tech":>10s}')
print('-' * 45)
for p_idx in range(6):
    action = mapper.get_auction_action(p_idx, company, ref_price, config)
    tech_idx = np.argmax(action[3:6])
    tech_names = ['onshore', 'offshore', 'solar']
    name = mapper.auction_profile_names()[p_idx]
    print(f'{name:<15s} {action[0]:7.1f} {action[1]:5.2f} {action[2]*100:5.1f} {tech_names[tech_idx]:>10s}')

## 4. Run a Random-Policy Episode (Pre-training Baseline)

In [ ]:
# Run one episode with fully random actions to establish a floor
env = ETSEnvironment(config, seed=42)
agents_random = [QLearningAgent(i, seed=42+i) for i in range(n_agents)]

obs1, _ = env.reset(seed=42)
total_rewards = np.zeros(n_agents)
prices, green_fracs = [], []

for year in range(n_years):
    price_ma3 = env._compute_price_ma3()
    
    # Phase 1 (random)
    auction_actions = np.zeros((n_agents, 6), dtype=np.float32)
    a1_indices = np.zeros(n_agents, dtype=int)
    for i in range(n_agents):
        action, a1_idx = agents_random[i].select_auction_action(
            obs1[i], env.companies[i], price_ma3, config, epsilon=1.0)
        auction_actions[i] = action
        a1_indices[i] = a1_idx
    
    obs2, _ = env.step_auction(auction_actions)
    
    # Phase 2 (random)
    sec_actions = np.zeros((n_agents, 2), dtype=np.float32)
    for i in range(n_agents):
        action, _ = agents_random[i].select_secondary_action(
            obs2[i], env.companies[i], env._phase1_clearing_price,
            config, a1_idx=a1_indices[i], epsilon=1.0)
        sec_actions[i] = action
    
    obs1, rewards, terminated, _, info = env.step_secondary(sec_actions)
    total_rewards += rewards
    
    yl = info.get('year_log', {})
    prices.append(yl.get('clearing_price', 0))
    green_fracs.append([env.companies[i].green_frac for i in range(n_agents)])

print('Random policy episode results:')
print(f'  Total rewards: {["{:.2f}".format(r) for r in total_rewards]}')
print(f'  Final green fracs: {["{:.1f}%".format(env.companies[i].green_frac*100) for i in range(n_agents)]}')
print(f'  Price range: {min(prices):.1f} - {max(prices):.1f} EUR/t')

In [ ]:
# Plot random episode
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(prices, 'b-o', markersize=4)
axes[0].set_title('Clearing Price')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('EUR/t')
axes[0].grid(True, alpha=0.3)

green_arr = np.array(green_fracs)
for i in range(n_agents):
    axes[1].plot(green_arr[:, i], label=f'A{i+1}', alpha=0.7)
axes[1].set_title('Green Fraction')
axes[1].set_xlabel('Year')
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(True, alpha=0.3)

axes[2].bar(range(n_agents), total_rewards)
axes[2].set_title('Total Reward')
axes[2].set_xlabel('Agent')
axes[2].set_xticks(range(n_agents))
axes[2].set_xticklabels([f'A{i+1}' for i in range(n_agents)])
axes[2].grid(True, alpha=0.3)

plt.suptitle('Random Policy Baseline (1 Episode)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Train Q-Learning Agents

Train for 5,000 episodes (adjustable via config). This should take a few minutes.

In [ ]:
SEED = 42

# Run training
trained_agents, trained_config = train_qlearning(base_config, ql_config, seed=SEED)

## 6. Training Curves

In [ ]:
results_dir = trained_config.get('logging', {}).get('results_dir', 'results/qlearning/')
ql_df = pd.read_csv(os.path.join(results_dir, f'ql_training_log_s{SEED}.csv'))

print(f'Training log: {len(ql_df)} episodes')
print(f'Columns: {list(ql_df.columns)[:10]}...')
ql_df.tail(3)

In [ ]:
# Reward curves
window = 50
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle(f'Per-Agent Reward (smoothed, window={window})', fontsize=14)

archetypes = ['Coal/Fin', 'Coal/Green', 'Gas/Fin', 'Gas/Green',
              'Trans/Fin', 'Trans/Green', 'Green/Fin', 'Green/Green']

for i in range(n_agents):
    ax = axes[i // 4, i % 4]
    col = f'reward_A{i+1}'
    rewards = ql_df[col].astype(float).values
    smoothed = pd.Series(rewards).rolling(window).mean()
    ax.plot(smoothed, color='tab:blue', alpha=0.8)
    ax.fill_between(range(len(smoothed)),
                    pd.Series(rewards).rolling(window).quantile(0.25),
                    pd.Series(rewards).rolling(window).quantile(0.75),
                    alpha=0.2, color='tab:blue')
    ax.set_title(f'A{i+1}: {archetypes[i]}', fontsize=10)
    ax.set_xlabel('Episode')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Price and green fraction trajectories
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Price
prices = ql_df['clearing_price_last'].astype(float)
axes[0].plot(prices.rolling(window).mean(), color='tab:red', alpha=0.8)
axes[0].set_title('Clearing Price (smoothed)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('EUR/t')
axes[0].grid(True, alpha=0.3)

# Green fractions
for i in range(n_agents):
    col = f'green_frac_A{i+1}'
    gf = ql_df[col].astype(float)
    axes[1].plot(gf.rolling(window).mean(), label=f'A{i+1}', alpha=0.7)
axes[1].set_title('Green Fraction (smoothed)')
axes[1].set_xlabel('Episode')
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(True, alpha=0.3)

# Compliance
for i in range(n_agents):
    col = f'compliance_rate_A{i+1}'
    cr = ql_df[col].astype(float)
    axes[2].plot(cr.rolling(window).mean(), label=f'A{i+1}', alpha=0.7)
axes[2].set_title('Compliance Rate (smoothed)')
axes[2].set_xlabel('Episode')
axes[2].legend(fontsize=7, ncol=2)
axes[2].set_ylim(0, 1.05)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Q-Table Analysis

In [ ]:
# Load final Q-tables
qtable_path = os.path.join(results_dir, f'qtables_s{SEED}_final.pkl')
with open(qtable_path, 'rb') as f:
    qtables = pickle.load(f)

print(f'Loaded Q-tables for {len(qtables)} agents')
for key, qt in qtables.items():
    nonzero = np.count_nonzero(qt)
    total = qt.size
    print(f'  {key}: shape={qt.shape}, '
          f'non-zero={nonzero}/{total} ({nonzero/total*100:.1f}%), '
          f'Q range=[{qt.min():.3f}, {qt.max():.3f}]')

In [ ]:
# Q-table heatmaps: dominant auction profile per state
plot_qtable_heatmaps(qtables, n_agents)
plt.show()

In [ ]:
# Q-table heatmaps: dominant secondary profile per state
plot_secondary_heatmaps(qtables, n_agents)
plt.show()

In [ ]:
# Top-5 Q-values per agent
a1_names = ActionProfileMapper.auction_profile_names()
a2_names = ActionProfileMapper.secondary_profile_names()
disc = StateDiscretizer()

for i in range(n_agents):
    agent = trained_agents[i]
    top = agent.get_top_q_values(5)
    print(f'\nA{i+1} ({archetypes[i]}) — Top 5 Q-values:')
    for s, a1, a2, qval in top:
        bins = disc.index_to_bins(s)
        bin_labels = ['early/mid/late', 'low/med/high', 'dirty/mix/green',
                      'deficit/bal/surplus', 'none/some/heavy']
        state_desc = ', '.join([['early','mid','late'][bins[0]],
                                ['lowP','medP','highP'][bins[1]],
                                ['dirty','mixed','green'][bins[2]],
                                ['deficit','balanced','surplus'][bins[3]],
                                ['noCF','someCF','heavyCF'][bins[4]]])
        print(f'  Q={qval:+.3f} | state=({state_desc}) | '
              f'auction={a1_names[a1]} | secondary={a2_names[a2]}')

## 8. Strategy Frequency Analysis

In [ ]:
# Strategy frequency in last 500 episodes
plot_strategy_frequency(ql_df, n_agents, last_n=500)
plt.show()

## 9. Greedy Evaluation (100 Episodes)

In [ ]:
eval_results = evaluate_qlearning(
    trained_agents, trained_config, seed=SEED,
    n_eval_episodes=100
)

In [ ]:
# Evaluation results visualization
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# Rewards
mean_rew = eval_results['rewards'].mean(axis=0)
std_rew = eval_results['rewards'].std(axis=0)
axes[0].bar(range(n_agents), mean_rew, yerr=std_rew, capsize=3,
            color=['steelblue' if i%2==0 else 'forestgreen' for i in range(n_agents)])
axes[0].set_title('Eval: Mean Reward')
axes[0].set_xticks(range(n_agents))
axes[0].set_xticklabels([f'A{i+1}' for i in range(n_agents)])
axes[0].grid(True, alpha=0.3)

# Green fracs
mean_gf = eval_results['green_fracs'].mean(axis=0)
axes[1].bar(range(n_agents), mean_gf * 100,
            color=['steelblue' if i%2==0 else 'forestgreen' for i in range(n_agents)])
axes[1].set_title('Eval: Final Green %')
axes[1].set_xticks(range(n_agents))
axes[1].set_xticklabels([f'A{i+1}' for i in range(n_agents)])
axes[1].grid(True, alpha=0.3)

# Compliance
mean_comp = eval_results['compliance'].mean(axis=0)
axes[2].bar(range(n_agents), mean_comp * 100,
            color=['steelblue' if i%2==0 else 'forestgreen' for i in range(n_agents)])
axes[2].set_title('Eval: Compliance Rate %')
axes[2].set_xticks(range(n_agents))
axes[2].set_xticklabels([f'A{i+1}' for i in range(n_agents)])
axes[2].set_ylim(0, 105)
axes[2].grid(True, alpha=0.3)

# Price distribution
axes[3].hist(eval_results['prices'], bins=20, color='tab:red', alpha=0.7)
axes[3].set_title('Eval: Final Clearing Price Distribution')
axes[3].set_xlabel('EUR/t')
axes[3].grid(True, alpha=0.3)

plt.suptitle('Greedy Evaluation Results (100 Episodes)', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Comparison with PPO/HAPPO (if available)

Load PPO training logs and overlay the learning curves.

In [ ]:
# Try to load PPO results for comparison
ppo_dir = 'results/'
ppo_log_path = os.path.join(ppo_dir, f'training_log_s{SEED}.csv')

if os.path.exists(ppo_log_path):
    ppo_df = pd.read_csv(ppo_log_path)
    print(f'PPO log loaded: {len(ppo_df)} episodes')
    HAS_PPO = True
else:
    print(f'PPO log not found at {ppo_log_path} - comparison plots will show Q-learning only.')
    print('Run PPO training first, then re-run this cell.')
    ppo_df = None
    HAS_PPO = False

In [ ]:
# Reward comparison
plot_reward_comparison(ql_df, ppo_df, n_agents)
plt.show()

In [ ]:
# Price comparison
plot_price_comparison(ql_df, ppo_df)
plt.show()

In [ ]:
# Green fraction comparison
plot_green_comparison(ql_df, ppo_df, n_agents)
plt.show()

In [ ]:
# Compliance comparison
plot_compliance_comparison(ql_df, ppo_df, n_agents)
plt.show()

## 11. Summary Statistics

In [ ]:
# Final summary table
last_500 = ql_df.tail(500)

summary_data = []
for i in range(n_agents):
    row = {
        'Agent': f'A{i+1}',
        'Archetype': archetypes[i],
        'Avg Reward': last_500[f'reward_A{i+1}'].astype(float).mean(),
        'Final Green %': last_500[f'green_frac_A{i+1}'].astype(float).mean() * 100,
        'Compliance %': last_500[f'compliance_rate_A{i+1}'].astype(float).mean() * 100,
        'Avg Shortfall': last_500[f'shortfall_A{i+1}'].astype(float).mean(),
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print('Q-Learning Performance (last 500 training episodes):')
print(summary_df.to_string(index=False, float_format='%.2f'))

if HAS_PPO:
    print('\n--- For comparison, PPO (last 500 episodes): ---')
    ppo_last = ppo_df.tail(500)
    for i in range(n_agents):
        col_r = f'reward_A{i+1}'
        col_g = f'green_frac_A{i+1}'
        if col_r in ppo_last.columns:
            ppo_rew = ppo_last[col_r].astype(float).mean()
            ppo_gf = ppo_last[col_g].astype(float).mean() * 100 if col_g in ppo_last.columns else 0
            print(f'  A{i+1}: reward={ppo_rew:.2f}, green={ppo_gf:.1f}%')

---

**Key Takeaways:**

- Q-learning with 243 states and 24 action profiles provides a tractable baseline
- The discretization necessarily loses information compared to PPO's continuous observations
- Comparison with PPO/HAPPO shows how much continuous-action deep RL gains over tabular methods
- This baseline helps quantify the "value of sophistication" in agent design for ETS markets